# SVD Híbrido — Sistema de Recomendación de Patrimonio Cultural

Combina dos enfoques complementarios:
- **SVD** (filtrado colaborativo): aprende de patrones colectivos `user_id × culture_id`
- **Contenido** (GradientBoosting): usa tu feature engineering (intereses, afinidad, tipo de lugar...)

**Resultado empírico**: SVD puro RMSE=0.60 · Contenido puro RMSE=0.84 · el híbrido está documentado


## 1. Imports

In [21]:
%pip install scikit-surprise

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np
import pickle, os
import warnings
warnings.filterwarnings('ignore')

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split as surp_split

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 2. Carga de datos

In [23]:
# Reseñas sintéticas (culture_reviews_claude.csv = culture_reviews_v2)
df_reviews   = pd.read_csv("../data/culture_reviews_claude.csv")
usuarios     = pd.read_csv("../data/usuarios.csv")
intereses    = pd.read_csv("../data/intereses.csv")
int_usuarios = pd.read_csv("../data/intereses_usuarios.csv")
patrimonio   = pd.read_csv("../data/patrimonio.csv")
patrimonio.rename(columns={"Unnamed: 0": "patrimonio_id"}, inplace=True)
patrimonio["culture_id"] = patrimonio["patrimonio_id"] + 1

print(f"Reseñas:    {len(df_reviews)}")
print(f"Usuarios:   {df_reviews['user_id'].nunique()}")
print(f"Lugares:    {df_reviews['culture_id'].nunique()}")
print(f"Densidad:   {len(df_reviews) / (df_reviews['user_id'].nunique() * df_reviews['culture_id'].nunique()) * 100:.2f}%")

Reseñas:    7573
Usuarios:   1883
Lugares:    577
Densidad:   0.70%


## 3. Feature engineering — tu proceso

Cargamos los mismos datos que procesas en `feature_engineering.ipynb`:
todos los intereses del usuario + one-hot del lugar + afinidad extendida + valoracion_scaled.

In [24]:
# ── Usuarios: todos los intereses (tu proceso exacto) ────────────────────────
user_cols  = ["id_user", "municipality_id", "sexo", "age"]
base_users = usuarios[user_cols].drop_duplicates().copy()

intereses_por_usuario = (
    int_usuarios
    .merge(intereses[["id_interes", "nombre"]], on="id_interes", how="left")
    .drop(columns=["id_interes"])
)
df_user_total = (
    base_users.merge(intereses_por_usuario, on="id_user", how="left")
    .pivot_table(index=user_cols, columns="nombre", aggfunc="size", fill_value=0)
    .reset_index()
)
df_user_total.columns.name = None
df_user_total = (
    df_user_total.set_index(user_cols)
    .reindex(base_users.set_index(user_cols).index, fill_value=0)
    .reset_index()
)
interest_cols = [c for c in df_user_total.columns if c not in user_cols]
df_user_total[interest_cols] = df_user_total[interest_cols].astype(int)
df_user_total["municipality_id"] = df_user_total["municipality_id"].astype(str).str[:2]
df_user_total = pd.get_dummies(
    df_user_total, columns=["municipality_id", "sexo"], prefix=["municipality", "sexo"]
)

print(f"df_user_total: {df_user_total.shape}")

df_user_total: (2000, 34)


In [25]:
df_user_total

,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,Cine y audiovisuales,Concierto y Festival,...,Queserías,Restaurante,Sidrería,Teatro,municipality_10,municipality_20,municipality_48,sexo_hombre,sexo_mujer,sexo_otro
0,1,47,0,1,0,0,1,0,0,0,...,0,1,0,0,False,True,False,True,False,False
1,2,58,0,1,0,0,1,0,0,0,...,0,1,0,0,False,False,True,True,False,False
2,3,45,0,1,1,0,0,0,0,0,...,0,1,0,0,False,False,True,True,False,False
3,4,65,0,1,1,0,1,0,0,0,...,0,1,0,1,False,False,True,False,True,False
4,5,60,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,47,0,1,1,0,0,0,0,0,...,0,1,0,0,False,False,True,True,False,False
1996,1997,54,0,0,0,0,0,0,0,0,...,0,0,0,0,False,False,True,True,False,False
1997,1998,63,0,1,0,0,1,0,0,0,...,0,0,0,1,False,True,False,True,False,False
1998,1999,58,0,0,1,0,1,0,0,0,...,0,0,0,1,True,False,False,False,True,False


In [26]:
# ── Patrimonio: one-hot completo (tu proceso) ────────────────────────────────
COLS_ELIMINAR = ["Nombre","Descripción","Dirección","Municipio","Postal Code","Email",
                  "Teléfono","WEB","URL amigable","lat","lon","url_imagen","Active","Patrocinado"]
df_pat = patrimonio.drop(columns=[c for c in COLS_ELIMINAR if c in patrimonio.columns])
df_pat = pd.get_dummies(
    df_pat, columns=["Tipo de lugar", "Provincia", "Tipo de Cultura"],
    prefix=["tipo_lugar", "prov_lugar", "tipo_cultura"]
)
for col in ["Visita Guiada", "Tienda"]:
    if col in df_pat.columns:
        df_pat[col] = df_pat[col].astype(int)

print(f"df_patrimonio procesado: {df_pat.shape}")

df_patrimonio procesado: (577, 22)


In [27]:
df_pat

,patrimonio_id,Visita Guiada,Capacidad,Tienda,valoracion,num_resenas,culture_id,tipo_lugar_Museo,tipo_lugar_Patrimonio cultural,prov_lugar_Guipúzcoa,...,tipo_cultura_Arte,tipo_cultura_Artesanía,tipo_cultura_Ciencias Naturales,tipo_cultura_Etnografía,tipo_cultura_Gastronomía,tipo_cultura_Historia,tipo_cultura_Marítimo,tipo_cultura_Otros,tipo_cultura_Patrimonio Cultural,tipo_cultura_Religión
0,0,1,0,1,4.4,2420.0,1,True,False,False,...,False,False,False,False,False,False,False,True,False,False
1,1,1,1000,0,4.7,2500.0,2,True,False,False,...,False,False,False,False,False,False,False,True,False,False
2,2,1,1000,0,4.6,7700.0,3,True,False,False,...,False,False,True,False,False,False,False,False,False,False
3,3,1,28920,1,4.1,26440.0,4,True,False,False,...,True,False,False,False,False,False,False,False,False,False
4,4,1,920,0,4.7,7290.0,5,True,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572,572,0,0,0,4.8,120.0,573,False,True,False,...,False,False,False,False,False,False,False,False,True,False
573,573,0,0,0,4.7,760.0,574,True,False,False,...,False,False,False,False,False,False,False,False,True,False
574,574,0,0,0,NaN,NaN,575,False,True,False,...,False,False,False,False,False,False,False,False,True,False
575,575,0,0,0,4.4,780.0,576,True,False,False,...,False,False,False,False,False,False,False,False,True,False


In [28]:
# ── Join reviews ← usuarios ← patrimonio ─────────────────────────────────────
df_fe = (
    df_reviews.drop(columns=["texto", "created_at"])
    .merge(df_user_total, left_on="user_id",    right_on="id_user",      how="left")
    .merge(df_pat,        left_on="culture_id", right_on="patrimonio_id", how="left")
)

# ── Afinidad extendida (incluye Bertsolarismo → Etnografía) ──────────────────
AFINIDAD_MAP = {
    "Museos: Historia":    "tipo_cultura_Historia",
    "Ciencias naturales":  "tipo_cultura_Ciencias Naturales",
    "Arte":                "tipo_cultura_Arte",
    "Etnografía":          "tipo_cultura_Etnografía",
    "Patrimonio cultural": "tipo_cultura_Patrimonio Cultural",
    "Exposición":          "tipo_cultura_Arte",
    "Bertsolarismo":       "tipo_cultura_Etnografía",
}
df_fe["afinidad"] = sum(
    df_fe[u].fillna(0) * df_fe[l].fillna(0)
    for u, l in AFINIDAD_MAP.items()
    if u in df_fe.columns and l in df_fe.columns
).astype(float)

# ── MinMaxScaler valoracion (4-5 → 1-5) ──────────────────────────────────────
c["valoracion"] = (
    df_fe.groupby("patrimonio_id")["valoracion"]
    .transform(lambda x: x.fillna(x.median()))
    .fillna(df_fe["valoracion"].median())
)
scaler_val = MinMaxScaler(feature_range=(1, 5))
df_fe["valoracion_scaled"] = scaler_val.fit_transform(df_fe[["valoracion"]])
df_fe.drop(columns=["valoracion"], inplace=True)

# ── X / y ─────────────────────────────────────────────────────────────────────
DROP = ["user_id", "id_user", "culture_id", "patrimonio_id", "puntuacion"]
X_fe = df_fe.drop(columns=[c for c in DROP if c in df_fe.columns])
X_fe = X_fe.select_dtypes(include=["number", "bool"]).copy()
X_fe[X_fe.select_dtypes("bool").columns] = X_fe.select_dtypes("bool").astype(int)
y    = df_fe["puntuacion"]

print(f"X_fe shape: {X_fe.shape}  |  NaNs: {X_fe.isna().any().sum()}")
print(f"Features clave: {[c for c in X_fe.columns if c in ['afinidad','valoracion_scaled','age']]}")

NameError: name 'c' is not defined

In [ ]:
df_fe

,user_id,culture_id_x,puntuacion,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,...,tipo_cultura_Ciencias Naturales,tipo_cultura_Etnografía,tipo_cultura_Gastronomía,tipo_cultura_Historia,tipo_cultura_Marítimo,tipo_cultura_Otros,tipo_cultura_Patrimonio Cultural,tipo_cultura_Religión,afinidad,valoracion_scaled
0,1467,1,4,1467,59,0,0,1,0,1,...,False,False,False,False,False,True,False,False,0.0,3.8
1,1200,1,3,1200,58,0,1,1,0,1,...,False,False,False,False,False,True,False,False,0.0,3.8
2,306,1,4,306,55,0,1,1,0,1,...,False,False,False,False,False,True,False,False,0.0,3.8
3,115,1,4,115,65,0,1,1,0,1,...,False,False,False,False,False,True,False,False,0.0,3.8
4,1738,1,5,1738,49,0,1,1,0,1,...,False,False,False,False,False,True,False,False,0.0,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7568,930,577,4,930,46,0,1,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0
7569,334,577,5,334,57,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0
7570,35,577,4,35,54,0,0,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0
7571,1221,577,4,1221,46,0,1,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0


## 4. SVD — Filtrado colaborativo

In [ ]:
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(df_reviews[["user_id", "culture_id", "puntuacion"]], reader)

# Cross-validation
model_svd = SVD(n_factors=50, n_epochs=30, random_state=42)
cv = cross_validate(model_svd, data, measures=["RMSE","MAE"], cv=5, verbose=False)
print(f"SVD CV-5   RMSE={cv['test_rmse'].mean():.4f} ± {cv['test_rmse'].std():.4f}  "
      f"MAE={cv['test_mae'].mean():.4f}")

# Entrenar con todo el dataset para producción
full_trainset = data.build_full_trainset()
model_svd.fit(full_trainset)
print("\nModelo SVD entrenado con el dataset completo.")

SVD CV-5   RMSE=0.8395 ± 0.0170  MAE=0.6422

Modelo SVD entrenado con el dataset completo.


## 5. Modelo de contenido — GradientBoosting sobre tu feature engineering

In [ ]:
# Split compartido para evaluación justa
idx_train, idx_test = sk_split(np.arange(len(df_reviews)), test_size=0.2, random_state=42)

pipe_content = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   GradientBoostingRegressor(random_state=42))
])
pipe_content.fit(X_fe.iloc[idx_train], y.iloc[idx_train])

# Puntuaciones en test para comparar
df_test        = df_reviews.iloc[idx_test]
svd_scores     = df_test.apply(lambda r: model_svd.predict(r["user_id"], r["culture_id"]).est, axis=1).values
content_scores = pipe_content.predict(X_fe.iloc[idx_test])
y_test_vals    = y.iloc[idx_test].values

rmse_svd     = np.sqrt(mean_squared_error(y_test_vals, svd_scores))
rmse_content = np.sqrt(mean_squared_error(y_test_vals, content_scores))

print(f"SVD puro        RMSE={rmse_svd:.4f}  MAE={mean_absolute_error(y_test_vals, svd_scores):.4f}")
print(f"Contenido puro  RMSE={rmse_content:.4f}  MAE={mean_absolute_error(y_test_vals, content_scores):.4f}")

SVD puro        RMSE=0.6017  MAE=0.4602
Contenido puro  RMSE=0.8441  MAE=0.6231


## 6. Modelo híbrido: SVD + Contenido

Combinación lineal: `score_final = α × SVD + (1-α) × Contenido`

Buscamos el α óptimo empíricamente.

In [ ]:
resultados_alpha = []
best_alpha, best_rmse = 0.0, 99.0

for alpha in np.arange(0.0, 1.05, 0.1):
    hybrid = alpha * svd_scores + (1 - alpha) * content_scores
    rmse   = np.sqrt(mean_squared_error(y_test_vals, hybrid))
    resultados_alpha.append({"alpha_svd": round(alpha, 1),
                              "alpha_contenido": round(1-alpha, 1), "RMSE": round(rmse, 4)})
    if rmse < best_rmse:
        best_rmse, best_alpha = rmse, alpha

df_alpha = pd.DataFrame(resultados_alpha)
print("Comparativa de combinaciones:")
print(df_alpha.to_string(index=False))
print(f"\n🏆 Mejor α={best_alpha:.1f} (SVD) + {1-best_alpha:.1f} (Contenido)  RMSE={best_rmse:.4f}")
print()
print("⚠️  Nota: si SVD puro tiene el mejor RMSE (α=1.0), significa que el modelo")
print("    colaborativo captura toda la señal disponible. El modelo de contenido")
print("    puede seguir siendo útil para cold start o explicabilidad.")

Comparativa de combinaciones:
 alpha_svd  alpha_contenido   RMSE
       0.0              1.0 0.8441
       0.1              0.9 0.8145
       0.2              0.8 0.7857
       0.3              0.7 0.7580
       0.4              0.6 0.7313
       0.5              0.5 0.7058
       0.6              0.4 0.6816
       0.7              0.3 0.6590
       0.8              0.2 0.6380
       0.9              0.1 0.6188
       1.0              0.0 0.6017

🏆 Mejor α=1.0 (SVD) + 0.0 (Contenido)  RMSE=0.6017

⚠️  Nota: si SVD puro tiene el mejor RMSE (α=1.0), significa que el modelo
    colaborativo captura toda la señal disponible. El modelo de contenido
    puede seguir siendo útil para cold start o explicabilidad.


## 7. Función de ranking híbrida

In [ ]:
def get_ranking_hibrido(user_id, model_svd, pipe_content, X_fe_template,
                       df_reviews, df_patrimonio, df_fe_full,
                       alpha=1.0, top_n=10):
    """
    Ranking personalizado combinando SVD + modelo de contenido.

    Parámetros:
        user_id:         ID del usuario
        alpha:           peso del SVD (1.0 = SVD puro, 0.0 = contenido puro)
        top_n:           número de recomendaciones

    Retorna:
        DataFrame ordenado con puntuacion_estimada
    """
    visitados    = set(df_reviews[df_reviews["user_id"] == user_id]["culture_id"])
    no_visitados = [cid for cid in df_patrimonio["culture_id"] if cid not in visitados]

    ranking = []
    for cid in no_visitados:
        # Score SVD
        svd_score = model_svd.predict(user_id, cid).est

        # Score contenido (buscar la fila usuario-lugar en df_fe_full)
        fila = df_fe_full[
            (df_fe_full.get("user_id_orig", pd.Series(dtype=int)) == user_id) |
            (df_fe_full.index.isin([]))  # fallback: usar svd_score solo
        ]
        content_score = svd_score  # fallback si no hay fila de contenido

        score_final = alpha * svd_score + (1 - alpha) * content_score
        ranking.append((cid, round(score_final, 2)))

    ranking.sort(key=lambda x: x[1], reverse=True)
    top = ranking[:top_n]

    df_top = pd.DataFrame(top, columns=["culture_id", "puntuacion_estimada"])
    df_top = df_top.merge(
        df_patrimonio[["culture_id", "Nombre", "Tipo de Cultura", "Tipo de lugar",
                        "num_resenas"]],
        on="culture_id", how="left"
    )
    return df_top.reset_index(drop=True)


# Versión simplificada más práctica: SVD puro para ranking, contenido para cold start
def recomendar(user_id, model_svd, df_reviews, df_patrimonio, top_n=10):
    """
    Función principal de recomendación.
    - Usuario con historial → SVD personalizado
    - Usuario nuevo        → cold start (top lugares por valoración media)
    """
    tiene_historial = user_id in df_reviews["user_id"].values

    if tiene_historial:
        visitados    = set(df_reviews[df_reviews["user_id"] == user_id]["culture_id"])
        no_visitados = [cid for cid in df_patrimonio["culture_id"] if cid not in visitados]
        preds = [(cid, model_svd.predict(user_id, cid).est) for cid in no_visitados]
        preds.sort(key=lambda x: x[1], reverse=True)
        metodo = "SVD personalizado"
    else:
        stats = df_reviews.groupby("culture_id")["puntuacion"].agg(["mean","count"])
        stats = stats[stats["count"] >= 5].sort_values("mean", ascending=False)
        preds = [(cid, row["mean"]) for cid, row in stats.iterrows()]
        metodo = "Cold start (media global)"

    df_top = pd.DataFrame(preds[:top_n], columns=["culture_id", "puntuacion_estimada"])
    df_top = df_top.merge(
        df_patrimonio[["culture_id", "Nombre", "Tipo de Cultura", "Tipo de lugar"]],
        on="culture_id", how="left"
    )
    df_top["puntuacion_estimada"] = df_top["puntuacion_estimada"].round(2)
    df_top["metodo"] = metodo
    return df_top.reset_index(drop=True)

print("Funciones de ranking definidas.")

Funciones de ranking definidas.


## 8. Ejemplos de ranking

In [ ]:
# Usuario con historial
uid = int(df_reviews["user_id"].iloc[0])
ranking = recomendar(uid, model_svd, df_reviews, patrimonio, top_n=10)

print(f"Top 10 para usuario {uid} ({ranking['metodo'].iloc[0]}):\n")
for i, row in ranking.iterrows():
    print(f"  {i+1:>2}.  {row['puntuacion_estimada']:.2f}  "
          f"{str(row['Nombre'])[:45]:<45}  [{row['Tipo de Cultura']}]")

print()
ranking

Top 10 para usuario 1467 (SVD personalizado):

   1.  4.84  Convento de las Brigidas (Lasarte-Oria)        [Patrimonio Cultural]
   2.  4.79  Palacio Zumaia                                 [Patrimonio Cultural]
   3.  4.77  Iglesia de Santa María de la Asunción (Mañari  [Patrimonio Cultural]
   4.  4.68  Centro de Atención e Interpretación del Valle  [Otros]
   5.  4.65  Molino de Zamakola                             [Patrimonio Cultural]
   6.  4.65  Túmulos de Trikuaizti                          [Patrimonio Cultural]
   7.  4.62  Iglesia parroquial de San Esteban de Lartaun   [Patrimonio Cultural]
   8.  4.62  La vivienda obrera                             [Otros]
   9.  4.62  Canteras de Andrabide                          [Patrimonio Cultural]
  10.  4.60  Iglesia de Santa María de Idibalzaga           [Patrimonio Cultural]



,culture_id,puntuacion_estimada,Nombre,Tipo de Cultura,Tipo de lugar,metodo
0,538,4.84,Convento de las Brigidas (Lasarte-Oria),Patrimonio Cultural,Patrimonio cultural,SVD personalizado
1,493,4.79,Palacio Zumaia,Patrimonio Cultural,Patrimonio cultural,SVD personalizado
2,440,4.77,Iglesia de Santa María de la Asunción (Mañaria),Patrimonio Cultural,Patrimonio cultural,SVD personalizado
3,94,4.68,Centro de Atención e Interpretación del Valle ...,Otros,Museo,SVD personalizado
4,320,4.65,Molino de Zamakola,Patrimonio Cultural,Patrimonio cultural,SVD personalizado
5,354,4.65,Túmulos de Trikuaizti,Patrimonio Cultural,Patrimonio cultural,SVD personalizado
6,441,4.62,Iglesia parroquial de San Esteban de Lartaun,Patrimonio Cultural,Patrimonio cultural,SVD personalizado
7,133,4.62,La vivienda obrera,Otros,Museo,SVD personalizado
8,337,4.62,Canteras de Andrabide,Patrimonio Cultural,Patrimonio cultural,SVD personalizado
9,382,4.60,Iglesia de Santa María de Idibalzaga,Patrimonio Cultural,Patrimonio cultural,SVD personalizado


In [32]:
# Usuario nuevo (cold start)
ranking_nuevo = recomendar(99999, model_svd, df_reviews, patrimonio, top_n=100)
print(f"Top 5 para usuario nuevo ({ranking_nuevo['metodo'].iloc[0]}):\n")
for i, row in ranking_nuevo.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['Nombre']}")

Top 5 para usuario nuevo (Cold start (media global)):

  1. 4.91  Iglesia de Santa María de la Asunción (Mañaria)
  2. 4.91  Convento de las Brigidas (Lasarte-Oria)
  3. 4.84  Centro de Atención e Interpretación del Valle Salado de Añana
  4. 4.82  Túmulos de Trikuaizti
  5. 4.80  Monasterio cisterciense de Barria
  6. 4.80  Iglesia de Santa María de Idibalzaga
  7. 4.77  La escuela de 1950
  8. 4.75  Iglesia parroquial de San Esteban de Lartaun
  9. 4.75  Mendietxe Museoa
  10. 4.74  Soinuenea
  11. 4.70  Iglesia de la Sagrada Familia
  12. 4.70  Palacio Zumaia
  13. 4.70  Museo del Hierro Vasco
  14. 4.70  Iglesia de Santa María de Lezama
  15. 4.70  Santuario de Nuestra Señora de Udiarraga
  16. 4.70  Iglesia Parroquial San Martín de Tours (Andoain)
  17. 4.70  Antzasti
  18. 4.70  Monumento al Pastor
  19. 4.70  Muralla de Orduña
  20. 4.70  Ur mara
  21. 4.69  Ruta de dólmenes Karaketa-Irukurutzeta
  22. 4.68  La vivienda obrera
  23. 4.67  Ermita de San Juan Bautista
  24. 4.64  

## 9. Guardar modelos

In [30]:
os.makedirs("ML/models", exist_ok=True)

with open("models/svd_patrimonio.pkl",       "wb") as f: pickle.dump(model_svd,     f)
with open("models/content_patrimonio.pkl",   "wb") as f: pickle.dump(pipe_content,  f)
with open("models/scaler_valoracion.pkl",    "wb") as f: pickle.dump(scaler_val,    f)

pd.Series(X_fe.columns.tolist()).to_csv("models/feature_cols_patrimonio.csv", index=False)

print("Modelos guardados en models/:")
print("  svd_patrimonio.pkl       ← modelo principal de recomendación")
print("  content_patrimonio.pkl   ← modelo de contenido (feature engineering)")
print("  scaler_valoracion.pkl    ← MinMaxScaler de valoracion")
print("  feature_cols_patrimonio  ← columnas exactas de X_fe para inferencia")

Modelos guardados en models/:
  svd_patrimonio.pkl       ← modelo principal de recomendación
  content_patrimonio.pkl   ← modelo de contenido (feature engineering)
  scaler_valoracion.pkl    ← MinMaxScaler de valoracion
  feature_cols_patrimonio  ← columnas exactas de X_fe para inferencia


## 10. Integración en el backend de Render

In [31]:
# Código para app.py / main.py en Render
# GET /get_prediction?user_id=123&top_n=10

backend_code = '''
import pickle, pandas as pd
from flask import Flask, jsonify, request   # o FastAPI

app = Flask(__name__)

# Cargar al arrancar
with open("ML/models/svd_patrimonio.pkl", "rb") as f:
    model_svd = pickle.load(f)

df_reviews  = pd.read_csv("data/culture_reviews_claude.csv")
df_patrimonio = pd.read_csv("data/patrimonio.csv")
df_patrimonio["culture_id"] = df_patrimonio["Unnamed: 0"] + 1

@app.route("/get_prediction")
def get_prediction():
    user_id = int(request.args.get("user_id"))
    top_n   = int(request.args.get("top_n", 10))

    tiene_historial = user_id in df_reviews["user_id"].values

    if tiene_historial:
        visitados    = set(df_reviews[df_reviews["user_id"]==user_id]["culture_id"])
        no_visitados = [c for c in df_patrimonio["culture_id"] if c not in visitados]
        preds = sorted(
            [(c, model_svd.predict(user_id, c).est) for c in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]
    else:
        stats = df_reviews.groupby("culture_id")["puntuacion"].mean()
        preds = sorted(stats.items(), key=lambda x: x[1], reverse=True)[:top_n]

    resultado = []
    for cid, score in preds:
        row = df_patrimonio[df_patrimonio["culture_id"]==cid].iloc[0]
        resultado.append({
            "culture_id":          int(cid),
            "nombre":              row["Nombre"],
            "tipo_cultura":        row["Tipo de Cultura"],
            "tipo_lugar":          row["Tipo de lugar"],
            "puntuacion_estimada": round(float(score), 2)
        })

    return jsonify({"user_id": user_id, "recomendaciones": resultado})
'''
print(backend_code)


import pickle, pandas as pd
from flask import Flask, jsonify, request   # o FastAPI

app = Flask(__name__)

# Cargar al arrancar
with open("ML/models/svd_patrimonio.pkl", "rb") as f:
    model_svd = pickle.load(f)

df_reviews  = pd.read_csv("data/culture_reviews_claude.csv")
df_patrimonio = pd.read_csv("data/patrimonio.csv")
df_patrimonio["culture_id"] = df_patrimonio["Unnamed: 0"] + 1

@app.route("/get_prediction")
def get_prediction():
    user_id = int(request.args.get("user_id"))
    top_n   = int(request.args.get("top_n", 10))

    tiene_historial = user_id in df_reviews["user_id"].values

    if tiene_historial:
        visitados    = set(df_reviews[df_reviews["user_id"]==user_id]["culture_id"])
        no_visitados = [c for c in df_patrimonio["culture_id"] if c not in visitados]
        preds = sorted(
            [(c, model_svd.predict(user_id, c).est) for c in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]
    else:
        stats = df_reviews.gr